# LLM-Prop: T5 Model Evaluation & Property Prediction Benchmark

This notebook evaluates the fine-tuned T5 sequence-to-sequence model on held-out materials dataset splits.
- **Classification Targets**: Electronic Type (Metal vs Non-Metal) & Thermodynamic Stability
- **Regression Targets**: Band Gap (eV), Formation Energy (eV/atom), Mass Density (g/cm³)
- **Metrics**: Accuracy, F1 Score, Confusion Matrix, ROC-AUC Curve, MAE, RMSE, R²

In [ ]:
import sys
from pathlib import Path

# Add repository root
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from backend.utils.config import settings
from training.evaluate import evaluate_t5_model

## 1. Run Complete Benchmark Evaluation
Executes the full evaluation pipeline across test materials.

In [ ]:
results = evaluate_t5_model(test_sample_limit=150)
clf_metrics = results['classification_metrics']
reg_metrics = results['metrics']

print(f"Accuracy:  {clf_metrics['accuracy'] * 100:.2f}%")
print(f"F1 Score:  {clf_metrics['f1_score_weighted']:.4f}")
print(f"ROC-AUC:   {clf_metrics['roc_auc']:.4f}")

## 2. Visualize Confusion Matrix

In [ ]:
cm = np.array(clf_metrics['confusion_matrix'])
fig, ax = plt.subplots(figsize=(6, 5))
cax = ax.matshow(cm, cmap='Blues', alpha=0.85)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), va='center', ha='center', fontsize=14, weight='bold')
fig.colorbar(cax)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Non-Metal', 'Metal'])
ax.set_yticklabels(['Non-Metal', 'Metal'])
ax.set_xlabel('Predicted Label', fontsize=11, labelpad=10)
ax.set_ylabel('True Label', fontsize=11)
ax.set_title('T5 Material Classification: Confusion Matrix', fontsize=12, pad=15, weight='bold')
plt.show()

## 3. Visualize ROC Curve

In [ ]:
roc = clf_metrics['roc_curve']
plt.figure(figsize=(6, 5))
plt.plot(roc['fpr'], roc['tpr'], color='#2563eb', lw=2.5, label=f"T5 Model (AUC = {clf_metrics['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], color='#94a3b8', linestyle='--', label='Random Chance (AUC = 0.500)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('T5 Classification: Receiver Operating Characteristic (ROC)', weight='bold')
plt.legend(loc='lower right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

## 4. Continuous Property Regression Metrics

In [ ]:
reg_df = pd.DataFrame([
    {'Property': v['property'], 'Unit': v['unit'], 'MAE': v['mae'], 'RMSE': v['rmse'], 'R2': v['r2']}
    for v in reg_metrics.values()
])
reg_df